In [11]:
from oqd_core.interface.analog import MathAdd, MathNum

a = MathNum(value=1) + MathNum(value=2)

print(a)

s = a.model_dump_json()

b = MathAdd.model_validate_json(s)

print(b)

class_='MathAdd' expr1=MathNum(class_='MathNum', value=1) expr2=MathNum(class_='MathNum', value=2)
class_='MathAdd' expr1=MathNum(class_='MathNum', value=1) expr2=MathNum(class_='MathNum', value=2)


In [12]:
from oqd_compiler_infrastructure import Post, PrettyPrint

from oqd_core.analysis.analog.cfg import Accumulator, AnalogCFGBuilder
from oqd_core.analysis.analog.type_checker import AnalogTypeChecker
from oqd_core.frontend.analog import parse_analog

printer = Post(PrettyPrint())

with open("test.analog", mode="r", encoding="utf8") as f:
    source = f.read()

circuit = parse_analog(source)
cfg = AnalogCFGBuilder().run(circuit)
cfg = Accumulator()(cfg)
checker = AnalogTypeChecker(cfg)


In [13]:
# from oqd_core.frontend.analog import parse_analog

# program = "r = qreg(2) \n H_single = 0.5 %* %X %+ 0.5 %* %Z \n evolve(H_single, 1.0, r[0])"
# circuit = parse_analog(program)

In [14]:
import json

from oqd_core.analysis.analog.symbol_table import AnalogSymbolTableBuilder

symbol_analysis = AnalogSymbolTableBuilder(cfg, checker.dataflow_result)
symbol_table = symbol_analysis.symbol_table


symbol_table = str(symbol_table)
print(json.dumps(symbol_table, indent=2))

"in_env={0: {}, 1: {}, 26: {'r': SymbolBinding(lattice_type=<class 'oqd_core.analysis.analog.types.TQReg'>, target_dim=2, list_elem=None), 's': SymbolBinding(lattice_type=<class 'oqd_core.analysis.analog.types.TMReg'>, target_dim=3, list_elem=None), 'q0': SymbolBinding(lattice_type=<class 'oqd_core.analysis.analog.types.TTargetRef'>, target_dim=1, list_elem=None), 'q1': SymbolBinding(lattice_type=<class 'oqd_core.analysis.analog.types.TTargetRef'>, target_dim=1, list_elem=None), 'targets': SymbolBinding(lattice_type=TList(elem=<class 'oqd_core.analysis.analog.types.TQRef'>), target_dim=2, list_elem=SymbolBinding(lattice_type=<class 'oqd_core.analysis.analog.types.TTargetRef'>, target_dim=1, list_elem=None))}, 27: {'r': SymbolBinding(lattice_type=<class 'oqd_core.analysis.analog.types.TQReg'>, target_dim=2, list_elem=None), 's': SymbolBinding(lattice_type=<class 'oqd_core.analysis.analog.types.TMReg'>, target_dim=3, list_elem=None), 'q0': SymbolBinding(lattice_type=<class 'oqd_core.anal

In [15]:
from oqd_core.analysis.analog import (
    AnalogCFGBuilder,
    AnalogSymbolTableBuilder,
    AnalogTypeChecker,
)
from oqd_core.compiler.analog.passes.compile import compile_analog_circuit

with open("test.analog", mode="r", encoding="utf8") as f:
    source = f.read()

circuit = parse_analog(source)
cfg = AnalogCFGBuilder().run(circuit)
cfg = Accumulator()(cfg)
type_checker = AnalogTypeChecker(cfg)
dataflow_result = type_checker.dataflow_result
symbol_analysis = AnalogSymbolTableBuilder(cfg, dataflow_result)
symbol_table = symbol_analysis.symbol_table
circuit, cfg = compile_analog_circuit(circuit, cfg, symbol_table)

In [16]:
cfg.to_dict()

{0: {'register_id': 0,
  'stmts': [{}],
  'preds': [],
  'succs': [1],
  'exit_nodes': [],
  'edge_labels': {}},
 1: {'register_id': 1,
  'stmts': [{'class_': 'Declaration',
    'name': 'r',
    'value': {'class_': 'QuantumRegister', 'size': 2}},
   {'class_': 'Declaration',
    'name': 's',
    'value': {'class_': 'ModeRegister', 'size': 3}},
   {'class_': 'Declaration',
    'name': 'q0',
    'value': {'class_': 'Extract',
     'access': {'class_': 'Access', 'name': 'r'},
     'index': 0}},
   {'class_': 'Declaration',
    'name': 'q1',
    'value': {'class_': 'Extract',
     'access': {'class_': 'Access', 'name': 'r'},
     'index': 1}},
   {'class_': 'Declaration',
    'name': 'targets',
    'value': {'class_': 'AnalogList',
     'values': [{'class_': 'Access', 'name': 'q0'},
      {'class_': 'Access', 'name': 'q1'}]}},
   {'class_': 'Declaration',
    'name': 'pi',
    'value': {'class_': 'MathNum', 'value': 3.14159}},
   {'class_': 'Declaration',
    'name': 'tau',
    'value': {'

In [17]:
circuit

AnalogCircuit(class_='AnalogCircuit', statements=[Declaration(class_='Declaration', name='r', value=QuantumRegister(class_='QuantumRegister', size=2)), Declaration(class_='Declaration', name='s', value=ModeRegister(class_='ModeRegister', size=3)), Declaration(class_='Declaration', name='q0', value=Extract(class_='Extract', access=Access(class_='Access', name='r'), index=0)), Declaration(class_='Declaration', name='q1', value=Extract(class_='Extract', access=Access(class_='Access', name='r'), index=1)), Declaration(class_='Declaration', name='targets', value=AnalogList(class_='AnalogList', values=[Access(class_='Access', name='q0'), Access(class_='Access', name='q1')])), Declaration(class_='Declaration', name='pi', value=MathNum(class_='MathNum', value=3.14159)), Declaration(class_='Declaration', name='tau', value=MathMul(class_='MathMul', expr1=MathNum(class_='MathNum', value=2), expr2=Access(class_='Access', name='pi'))), Declaration(class_='Declaration', name='omega', value=MathVar(c

In [18]:
from oqd_core.analysis.utils import cfg_to_dot

dot = cfg_to_dot(cfg)
print(dot.source)
dot.render("cfg", format="png", cleanup=True)


digraph {
	0 [label="0: VisitableBaseModel\n"]
	0 -> 1
	1 [label="1: r = ...\n"]
	1 [label="1: s = ...\n"]
	1 [label="1: q0 = ...\n"]
	1 [label="1: q1 = ...\n"]
	1 [label="1: targets = ...\n"]
	1 [label="1: pi = ...\n"]
	1 [label="1: tau = ...\n"]
	1 [label="1: omega = ...\n"]
	1 [label="1: phase = ...\n"]
	1 [label="1: neg = ...\n"]
	1 [label="1: cubed = ...\n"]
	1 [label="1: sine = ...\n"]
	1 [label="1: cosed = ...\n"]
	1 [label="1: cplx = ...\n"]
	1 [label="1: X = ...\n"]
	1 [label="1: Y = ...\n"]
	1 [label="1: Z = ...\n"]
	1 [label="1: I = ...\n"]
	1 [label="1: C = ...\n"]
	1 [label="1: A = ...\n"]
	1 [label="1: J = ...\n"]
	1 [label="1: H_single = ...\n"]
	1 [label="1: H_pair = ...\n"]
	1 [label="1: H_rabi = ...\n"]
	1 [label="1: x = ...\n"]
	1 -> 26
	26 [label="26: BoolGreaterThan\n"]
	26 -> 27
	26 -> 28
	27 [label="27: Initialize\n"]
	27 -> 28
	28 [label="28: BoolEq\n"]
	28 -> 29
	28 -> 30
	29 [label="29: Evolve\n"]
	29 -> 31
	30 [label="30: Evolve\n"]
	30 -> 31
	31 [label="31: 

'cfg.png'

In [19]:
# from oqd_core.compiler.analog.passes.compile import compile_analog_circuit
# out = compile_analog_circuit(circuit)
# print(printer(out))

In [20]:
# cfg = AnalogCFGBuilder().run(circuit)
# out = json.dumps({node_id: node.to_dict() for node_id, node in cfg.items()}, indent=2)
# print(out)


# print(symbol_table)
# symbol_table = str(symbol_table)
# tree = symbol_table.model_dump_json(indent=2, serialize_as_any=True)
# print(json.dumps(cfg, indent=2))